In [3]:
!pip -q install transformers accelerate scikit-learn pandas

In [4]:
from google.colab import drive
drive.mount("/content/drive")

import os
base = "/content/drive/MyDrive/greek-news-nlp"
for root, dirs, files in os.walk(base):
    depth = root[len(base):].count(os.sep)
    if depth <= 2:
        print(root)
        for f in files:
            print("   ", f)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
/content/drive/MyDrive/greek-news-nlp
    el_train.jsonl
    el_val.jsonl
    el_test.jsonl
    greekbert_confusion_matrix.png
/content/drive/MyDrive/greek-news-nlp/greek-bert-news-classifier
    config.json
    training_args.bin
    tokenizer_config.json
    tokenizer.json
    model.safetensors


In [5]:
import pandas as pd

TEST_PATH = "/content/drive/MyDrive/greek-news-nlp/el_test.jsonl"
TEXT_COL, LABEL_COL = "text_clean", "GPT-IPTC-label"
MODEL_ID = "Qwen/Qwen2.5-1.5B-Instruct"
MAX_CHARS = 1200
N_SAMPLES = 100        # smoke test, μετά βάλε None για όλα τα 788
BATCH = 8

test = pd.read_json(TEST_PATH, lines=True)
if N_SAMPLES:
    test = test.sample(N_SAMPLES, random_state=42).reset_index(drop=True)
LABELS = sorted(test[LABEL_COL].astype(str).unique())
print(len(test), "samples,", len(LABELS), "labels")
print(LABELS)

100 samples, 17 labels
['arts, culture, entertainment and media', 'conflict, war and peace', 'crime, law and justice', 'disaster, accident and emergency incident', 'economy, business and finance', 'education', 'environment', 'health', 'human interest', 'labour', 'lifestyle and leisure', 'politics', 'religion', 'science and technology', 'society', 'sport', 'weather']


In [6]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

tok = AutoTokenizer.from_pretrained(MODEL_ID)
tok.padding_side = "left"
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID, torch_dtype=torch.float16, device_map="auto"
).eval()
print("loaded on", model.device)

config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

...

model.safetensors: reconstructing file:   0%|          |  0.00B / 3.09GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

loaded on cuda:0


In [8]:
import time

label_list = "\n".join(f"- {l}" for l in LABELS)

def build_prompt(text):
    messages = [
        {"role": "system", "content": "You are a precise news topic classifier."},
        {"role": "user", "content":
            f"Classify the following Greek news article into exactly one of these topics:\n"
            f"{label_list}\n\n"
            f"Article:\n{text[:MAX_CHARS]}\n\n"
            f"Answer with the topic name only, exactly as written in the list."},
    ]
    return tok.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

def parse(output):
    out = output.strip().lower()
    for l in sorted(LABELS, key=len, reverse=True):
        if l.lower() in out:
            return l
    return "UNPARSED"

torch.cuda.reset_peak_memory_stats()
preds, raw, times = [], [], []
texts = test[TEXT_COL].tolist()

for i in range(0, len(texts), BATCH):
    batch = [build_prompt(t) for t in texts[i:i+BATCH]]
    enc = tok(batch, return_tensors="pt", padding=True).to(model.device)
    t0 = time.time()
    with torch.no_grad():
        gen = model.generate(**enc, max_new_tokens=24, do_sample=False)
    dt = time.time() - t0
    outs = tok.batch_decode(gen[:, enc.input_ids.shape[1]:], skip_special_tokens=True)
    raw += outs
    preds += [parse(o) for o in outs]
    times += [dt / len(batch)] * len(batch)

print("done:", len(preds))
print(raw[:5])

done: 100
['technology', 'sport', 'disaster, accident and emergency incident', 'crime, law and justice', 'conflict, war and peace']


In [9]:
from sklearn.metrics import f1_score, accuracy_score

y_true = test[LABEL_COL].astype(str).tolist()
res = {
    "model": MODEL_ID,
    "mode": "zero-shot",
    "n": len(test),
    "accuracy": round(accuracy_score(y_true, preds), 4),
    "macro_f1": round(f1_score(y_true, preds, average="macro", labels=LABELS, zero_division=0), 4),
    "unparsed_rate": round(sum(p == "UNPARSED" for p in preds) / len(preds), 4),
    "sec_per_article": round(sum(times) / len(times), 3),
    "peak_vram_gb": round(torch.cuda.max_memory_allocated() / 1e9, 2),
}
print(res)

out = test.copy()
out["pred"], out["raw_output"] = preds, raw
out.to_csv("/content/drive/MyDrive/greek-news-nlp/zero_shot_qwen2.5-1.5b_n100.csv", index=False)
print("saved")

{'model': 'Qwen/Qwen2.5-1.5B-Instruct', 'mode': 'zero-shot', 'n': 100, 'accuracy': 0.45, 'macro_f1': 0.3531, 'unparsed_rate': 0.12, 'sec_per_article': 0.291, 'peak_vram_gb': 4.02}
saved


In [11]:
import string

VAL_PATH = "/content/drive/MyDrive/greek-news-nlp/el_val.jsonl"
val = pd.read_json(VAL_PATH, lines=True)

LABELS = sorted(val[LABEL_COL].astype(str).unique())      # σταθερή σειρά, 17 labels
CODES = list(string.ascii_uppercase[:len(LABELS)])         # A..Q
code2label = dict(zip(CODES, LABELS))

print(len(val), "val samples,", len(LABELS), "labels")
print(val[LABEL_COL].value_counts())

# κάθε γράμμα πρέπει να είναι ένα μόνο token
code_ids = []
for c in CODES:
    ids = tok.encode(c, add_special_tokens=False)
    assert len(ids) == 1, (c, ids)
    code_ids.append(ids[0])
print("code token ids:", code_ids)

787 val samples, 17 labels
GPT-IPTC-label
sport                                        134
politics                                     118
economy, business and finance                 99
arts, culture, entertainment and media        79
crime, law and justice                        63
human interest                                62
health                                        39
disaster, accident and emergency incident     36
education                                     31
conflict, war and peace                       26
labour                                        21
environment                                   18
science and technology                        18
society                                       17
religion                                      13
lifestyle and leisure                          8
weather                                        5
Name: count, dtype: int64
code token ids: [32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48]


In [12]:
import numpy as np, time

options = "\n".join(f"{c}. {l}" for c, l in zip(CODES, LABELS))

def build_prompt_codes(text):
    messages = [
        {"role": "system", "content": "You are a precise news topic classifier."},
        {"role": "user", "content":
            f"Read the Greek news article and choose the single best topic.\n\n"
            f"Topics:\n{options}\n\n"
            f"Article:\n{text[:MAX_CHARS]}\n\n"
            f"Answer with only the letter of the topic."},
    ]
    return tok.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

torch.cuda.reset_peak_memory_stats()
preds_v, times_v = [], []
texts_v = val[TEXT_COL].tolist()
code_ids_t = torch.tensor(code_ids, device=model.device)

for i in range(0, len(texts_v), BATCH):
    batch = [build_prompt_codes(t) for t in texts_v[i:i+BATCH]]
    enc = tok(batch, return_tensors="pt", padding=True).to(model.device)
    t0 = time.time()
    with torch.no_grad():
        logits = model(**enc, logits_to_keep=1).logits[:, -1, :]
    dt = time.time() - t0
    scores = logits[:, code_ids_t]                       # μόνο τα 17 γράμματα
    best = scores.argmax(dim=-1).tolist()
    preds_v += [LABELS[b] for b in best]
    times_v += [dt / len(batch)] * len(batch)

print("done:", len(preds_v))

done: 787


In [13]:
from sklearn.metrics import f1_score, accuracy_score, classification_report

y_val = val[LABEL_COL].astype(str).tolist()
res = {
    "model": MODEL_ID, "mode": "zero-shot, letter-constrained", "split": "val", "n": len(val),
    "accuracy": round(accuracy_score(y_val, preds_v), 4),
    "macro_f1": round(f1_score(y_val, preds_v, average="macro", labels=LABELS, zero_division=0), 4),
    "sec_per_article": round(sum(times_v) / len(times_v), 3),
    "peak_vram_gb": round(torch.cuda.max_memory_allocated() / 1e9, 2),
}
print(res)
print(classification_report(y_val, preds_v, labels=LABELS, zero_division=0))
print(pd.Series(preds_v).value_counts())

out = val.copy(); out["pred"] = preds_v
out.to_csv("/content/drive/MyDrive/greek-news-nlp/val_zero_shot_letters_qwen2.5-1.5b.csv", index=False)
print("saved")

{'model': 'Qwen/Qwen2.5-1.5B-Instruct', 'mode': 'zero-shot, letter-constrained', 'split': 'val', 'n': 787, 'accuracy': 0.0559, 'macro_f1': 0.0237, 'sec_per_article': 0.05, 'peak_vram_gb': 4.12}
                                           precision    recall  f1-score   support

   arts, culture, entertainment and media       0.14      0.01      0.02        79
                  conflict, war and peace       0.04      1.00      0.07        26
                   crime, law and justice       1.00      0.02      0.03        63
disaster, accident and emergency incident       0.00      0.00      0.00        36
            economy, business and finance       0.60      0.03      0.06        99
                                education       0.00      0.00      0.00        31
                              environment       0.00      0.00      0.00        18
                                   health       0.02      0.03      0.02        39
                           human interest       0.43      

In [14]:
# 1) ίδια 40 άρθρα με batch=1 (χωρίς padding)
p1 = []
for i in range(40):
    enc = tok(build_prompt_codes(texts_v[i]), return_tensors="pt").to(model.device)
    with torch.no_grad():
        lg = model(**enc, logits_to_keep=1).logits[0, -1, :]
    p1.append(LABELS[int(lg[code_ids_t].argmax())])

print("batch=1:", pd.Series(p1).value_counts().head(5).to_dict())
print("batch=8:", pd.Series(preds_v[:40]).value_counts().head(5).to_dict())
print("ίδιες προβλέψεις:", sum(a == b for a, b in zip(p1, preds_v[:40])), "/ 40")

# 2) τι θέλει πραγματικά να γράψει το μοντέλο στο πρώτο token (για 1 άρθρο)
enc = tok(build_prompt_codes(texts_v[0]), return_tensors="pt").to(model.device)
with torch.no_grad():
    lg = model(**enc, logits_to_keep=1).logits[0, -1, :]
print("top-5 tokens:", [tok.decode([t]) for t in lg.topk(5).indices.tolist()])

batch=1: {'conflict, war and peace': 34, 'politics': 4, 'health': 2}
batch=8: {'conflict, war and peace': 34, 'politics': 4, 'health': 2}
ίδιες προβλέψεις: 40 / 40
top-5 tokens: ['B', 'H', 'A', 'L', 'C']


In [15]:
label_list = "\n".join(f"- {l}" for l in LABELS)

first_ids = [tok.encode(l, add_special_tokens=False)[0] for l in LABELS]
print([tok.decode([i]) for i in first_ids])
assert len(set(first_ids)) == len(LABELS), "διπλά first tokens"
first_ids_t = torch.tensor(first_ids, device=model.device)

def build_prompt_names(text):
    messages = [
        {"role": "system", "content": "You are a precise news topic classifier."},
        {"role": "user", "content":
            f"Classify the following Greek news article into exactly one of these topics:\n"
            f"{label_list}\n\n"
            f"Article:\n{text[:MAX_CHARS]}\n\n"
            f"Answer with the topic name only, exactly as written in the list."},
    ]
    return tok.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

torch.cuda.reset_peak_memory_stats()
preds_n, times_n = [], []
for i in range(0, len(texts_v), BATCH):
    batch = [build_prompt_names(t) for t in texts_v[i:i+BATCH]]
    enc = tok(batch, return_tensors="pt", padding=True).to(model.device)
    t0 = time.time()
    with torch.no_grad():
        logits = model(**enc, logits_to_keep=1).logits[:, -1, :]
    dt = time.time() - t0
    best = logits[:, first_ids_t].argmax(dim=-1).tolist()
    preds_n += [LABELS[b] for b in best]
    times_n += [dt / len(batch)] * len(batch)

res = {
    "model": MODEL_ID, "mode": "zero-shot, first-token constrained", "split": "val", "n": len(val),
    "accuracy": round(accuracy_score(y_val, preds_n), 4),
    "macro_f1": round(f1_score(y_val, preds_n, average="macro", labels=LABELS, zero_division=0), 4),
    "sec_per_article": round(sum(times_n) / len(times_n), 3),
    "peak_vram_gb": round(torch.cuda.max_memory_allocated() / 1e9, 2),
}
print(res)
print(pd.Series(preds_n).value_counts())

['arts', 'conf', 'crime', 'dis', 'e', 'education', 'environment', 'health', 'human', 'lab', 'l', 'pol', 'rel', 'science', 's', 'sport', 'weather']
{'model': 'Qwen/Qwen2.5-1.5B-Instruct', 'mode': 'zero-shot, first-token constrained', 'split': 'val', 'n': 787, 'accuracy': 0.5095, 'macro_f1': 0.4276, 'sec_per_article': 0.05, 'peak_vram_gb': 4.11}
sport                                        232
arts, culture, entertainment and media       101
disaster, accident and emergency incident     77
crime, law and justice                        64
economy, business and finance                 59
science and technology                        44
politics                                      42
conflict, war and peace                       39
health                                        39
education                                     28
environment                                   28
lifestyle and leisure                         13
human interest                                12
religion         

In [16]:
import torch.nn.functional as F

lab_ids = [tok.encode(l, add_special_tokens=False) for l in LABELS]
K = max(len(x) for x in lab_ids) + 1
pad_id = tok.pad_token_id if tok.pad_token_id is not None else tok.eos_token_id

@torch.no_grad()
def score_labels(prompt_text):
    p_ids = tok(prompt_text, add_special_tokens=False).input_ids
    seqs = [p_ids + l for l in lab_ids]
    T = max(len(s) for s in seqs)
    input_ids = torch.full((len(seqs), T), pad_id)
    attn = torch.zeros((len(seqs), T), dtype=torch.long)
    for i, s in enumerate(seqs):
        input_ids[i, T-len(s):] = torch.tensor(s)
        attn[i, T-len(s):] = 1
    pos = (attn.cumsum(-1) - 1).clamp(min=0)
    out = model(input_ids=input_ids.to(model.device),
                attention_mask=attn.to(model.device),
                position_ids=pos.to(model.device),
                logits_to_keep=K, use_cache=False)
    logp = F.log_softmax(out.logits.float(), dim=-1)
    sums, means = [], []
    for i, l in enumerate(lab_ids):
        n = len(l)
        lp = sum(logp[i, K-n+m-1, l[m]].item() for m in range(n))
        sums.append(lp)
        means.append(lp / n)
    return sums, means

N_EVAL = 200
torch.cuda.reset_peak_memory_stats()
pred_sum, pred_mean = [], []
t0 = time.time()
for i in range(N_EVAL):
    s, m = score_labels(build_prompt_names(texts_v[i]))
    pred_sum.append(LABELS[int(np.argmax(s))])
    pred_mean.append(LABELS[int(np.argmax(m))])
    if (i + 1) % 50 == 0:
        print(i + 1, "done,", round(time.time() - t0), "s")

y_sub = y_val[:N_EVAL]
def mf1(p): return round(f1_score(y_sub, p, average="macro", labels=LABELS, zero_division=0), 4)
def acc(p): return round(accuracy_score(y_sub, p), 4)

print("first-token  :", acc(preds_n[:N_EVAL]), mf1(preds_n[:N_EVAL]))
print("full sum     :", acc(pred_sum), mf1(pred_sum))
print("full mean    :", acc(pred_mean), mf1(pred_mean))
print("sec/article  :", round((time.time() - t0) / N_EVAL, 2))
print("peak VRAM GB :", round(torch.cuda.max_memory_allocated() / 1e9, 2))
print(pd.Series(pred_mean).value_counts())

50 done, 193 s
100 done, 366 s
150 done, 556 s
200 done, 742 s
first-token  : 0.465 0.3557
full sum     : 0.465 0.3549
full mean    : 0.405 0.2695
sec/article  : 3.71
peak VRAM GB : 4.5
arts, culture, entertainment and media       47
disaster, accident and emergency incident    46
sport                                        32
conflict, war and peace                      16
economy, business and finance                16
crime, law and justice                       13
science and technology                       11
lifestyle and leisure                         7
politics                                      5
education                                     4
health                                        2
human interest                                1
Name: count, dtype: int64


In [17]:
TRAIN_PATH = "/content/drive/MyDrive/greek-news-nlp/el_train.jsonl"
train = pd.read_json(TRAIN_PATH, lines=True)

# 1 παράδειγμα ανά κατηγορία, σταθερό seed
shots = (train.groupby(LABEL_COL, group_keys=False)
              .apply(lambda g: g.sample(1, random_state=42))
              .set_index(LABEL_COL).loc[LABELS].reset_index())
SHOT_CHARS = 400
examples = "\n\n".join(
    f"Article: {r[TEXT_COL][:SHOT_CHARS]}\nTopic: {r[LABEL_COL]}" for _, r in shots.iterrows()
)

def build_prompt_fewshot(text):
    messages = [
        {"role": "system", "content": "You are a precise news topic classifier."},
        {"role": "user", "content":
            f"Classify Greek news articles into exactly one of these topics:\n{label_list}\n\n"
            f"Here is one example per topic:\n\n{examples}\n\n"
            f"Now classify this article.\nArticle: {text[:MAX_CHARS]}\n\n"
            f"Answer with the topic name only, exactly as written in the list."},
    ]
    return tok.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

print("prompt tokens:", len(tok(build_prompt_fewshot(texts_v[0])).input_ids))

BATCH_FS = 4
torch.cuda.reset_peak_memory_stats()
preds_fs, times_fs = [], []
for i in range(0, len(texts_v), BATCH_FS):
    batch = [build_prompt_fewshot(t) for t in texts_v[i:i+BATCH_FS]]
    enc = tok(batch, return_tensors="pt", padding=True).to(model.device)
    t0 = time.time()
    with torch.no_grad():
        logits = model(**enc, logits_to_keep=1).logits[:, -1, :]
    dt = time.time() - t0
    best = logits[:, first_ids_t].argmax(dim=-1).tolist()
    preds_fs += [LABELS[b] for b in best]
    times_fs += [dt / len(batch)] * len(batch)

res = {
    "model": MODEL_ID, "mode": "few-shot (1/class), first-token", "split": "val", "n": len(val),
    "accuracy": round(accuracy_score(y_val, preds_fs), 4),
    "macro_f1": round(f1_score(y_val, preds_fs, average="macro", labels=LABELS, zero_division=0), 4),
    "sec_per_article": round(sum(times_fs) / len(times_fs), 3),
    "peak_vram_gb": round(torch.cuda.max_memory_allocated() / 1e9, 2),
}
print(res)
print(pd.Series(preds_fs).value_counts())

out = val.copy(); out["pred"] = preds_fs
out.to_csv("/content/drive/MyDrive/greek-news-nlp/val_fewshot_qwen2.5-1.5b.csv", index=False)
print("saved")

/tmp/ipykernel_1232/4122381322.py:6: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda g: g.sample(1, random_state=42))


prompt tokens: 7083
{'model': 'Qwen/Qwen2.5-1.5B-Instruct', 'mode': 'few-shot (1/class), first-token', 'split': 'val', 'n': 787, 'accuracy': 0.4854, 'macro_f1': 0.3853, 'sec_per_article': 0.525, 'peak_vram_gb': 6.05}
sport                                        204
arts, culture, entertainment and media        78
human interest                                71
politics                                      62
conflict, war and peace                       47
disaster, accident and emergency incident     45
society                                       38
crime, law and justice                        38
science and technology                        37
economy, business and finance                 36
lifestyle and leisure                         31
health                                        30
environment                                   27
education                                     22
weather                                       14
religion                                       7

In [18]:
rows = [
    {"model": "Qwen2.5-1.5B-Instruct", "method": "zero-shot, free generation", "split": "test (smoke, n=100)", "accuracy": 0.45, "macro_f1": 0.3531, "sec_per_article": 0.291, "peak_vram_gb": 4.02, "notes": "12% unparsed"},
    {"model": "Qwen2.5-1.5B-Instruct", "method": "zero-shot, letter choice", "split": "val", "accuracy": 0.0559, "macro_f1": 0.0237, "sec_per_article": 0.05, "peak_vram_gb": 4.12, "notes": "collapsed to one letter"},
    {"model": "Qwen2.5-1.5B-Instruct", "method": "zero-shot, first-token", "split": "val", "accuracy": 0.5095, "macro_f1": 0.4276, "sec_per_article": 0.05, "peak_vram_gb": 4.11, "notes": ""},
    {"model": "Qwen2.5-1.5B-Instruct", "method": "zero-shot, full-label scoring (sum)", "split": "val (first 200)", "accuracy": 0.465, "macro_f1": 0.3549, "sec_per_article": 3.71, "peak_vram_gb": 4.5, "notes": "no gain over first-token (0.465 / 0.3557 on same 200)"},
    {"model": "Qwen2.5-1.5B-Instruct", "method": "few-shot 1/class, first-token", "split": "val", "accuracy": 0.4854, "macro_f1": 0.3853, "sec_per_article": 0.525, "peak_vram_gb": 6.05, "notes": "7083 prompt tokens"},
    {"model": "GreekBERT (fine-tuned)", "method": "fine-tuned classifier", "split": "val", "accuracy": 0.820, "macro_f1": 0.784, "sec_per_article": None, "peak_vram_gb": None, "notes": "reference"},
]
summary = pd.DataFrame(rows)
summary.to_csv("/content/drive/MyDrive/greek-news-nlp/benchmark_summary.csv", index=False)
summary

,model,method,split,accuracy,macro_f1,sec_per_article,peak_vram_gb,notes
0,Qwen2.5-1.5B-Instruct,"zero-shot, free generation","test (smoke, n=100)",0.4500,0.3531,0.291,4.02,12% unparsed
1,Qwen2.5-1.5B-Instruct,"zero-shot, letter choice",val,0.0559,0.0237,0.050,4.12,collapsed to one letter
2,Qwen2.5-1.5B-Instruct,"zero-shot, first-token",val,0.5095,0.4276,0.050,4.11,
3,Qwen2.5-1.5B-Instruct,"zero-shot, full-label scoring (sum)",val (first 200),0.4650,0.3549,3.710,4.50,no gain over first-token (0.465 / 0.3557 on sa...
4,Qwen2.5-1.5B-Instruct,"few-shot 1/class, first-token",val,0.4854,0.3853,0.525,6.05,7083 prompt tokens
5,GreekBERT (fine-tuned),fine-tuned classifier,val,0.8200,0.7840,NaN,NaN,reference


In [19]:
import gc
del model
gc.collect(); torch.cuda.empty_cache()

MODEL_ID = "Qwen/Qwen2.5-3B-Instruct"
tok = AutoTokenizer.from_pretrained(MODEL_ID)
tok.padding_side = "left"
model = AutoModelForCausalLM.from_pretrained(MODEL_ID, torch_dtype=torch.float16, device_map="auto").eval()

first_ids = [tok.encode(l, add_special_tokens=False)[0] for l in LABELS]
assert len(set(first_ids)) == len(LABELS)
first_ids_t = torch.tensor(first_ids, device=model.device)

torch.cuda.reset_peak_memory_stats()
preds_3b, times_3b = [], []
for i in range(0, len(texts_v), BATCH):
    batch = [build_prompt_names(t) for t in texts_v[i:i+BATCH]]
    enc = tok(batch, return_tensors="pt", padding=True).to(model.device)
    t0 = time.time()
    with torch.no_grad():
        logits = model(**enc, logits_to_keep=1).logits[:, -1, :]
    dt = time.time() - t0
    preds_3b += [LABELS[b] for b in logits[:, first_ids_t].argmax(dim=-1).tolist()]
    times_3b += [dt / len(batch)] * len(batch)

res = {
    "model": MODEL_ID, "mode": "zero-shot, first-token", "split": "val", "n": len(val),
    "accuracy": round(accuracy_score(y_val, preds_3b), 4),
    "macro_f1": round(f1_score(y_val, preds_3b, average="macro", labels=LABELS, zero_division=0), 4),
    "sec_per_article": round(sum(times_3b) / len(times_3b), 3),
    "peak_vram_gb": round(torch.cuda.max_memory_allocated() / 1e9, 2),
}
print(res)
print(pd.Series(preds_3b).value_counts())

config.json:   0%|          | 0.00/661 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

...

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

{'model': 'Qwen/Qwen2.5-3B-Instruct', 'mode': 'zero-shot, first-token', 'split': 'val', 'n': 787, 'accuracy': 0.6264, 'macro_f1': 0.5293, 'sec_per_article': 0.18, 'peak_vram_gb': 7.45}
sport                                        142
arts, culture, entertainment and media       114
politics                                      90
lifestyle and leisure                         78
economy, business and finance                 76
crime, law and justice                        61
education                                     55
health                                        44
environment                                   33
human interest                                25
society                                       23
science and technology                        15
disaster, accident and emergency incident     13
conflict, war and peace                        7
religion                                       7
weather                                        4
Name: count, dtype: int64


In [20]:
new = pd.DataFrame([{
    "model": "Qwen2.5-3B-Instruct", "method": "zero-shot, first-token", "split": "val",
    "accuracy": 0.6264, "macro_f1": 0.5293, "sec_per_article": 0.18, "peak_vram_gb": 7.45,
    "notes": "labour never predicted"
}])
summary = pd.concat([summary, new], ignore_index=True)
summary.to_csv("/content/drive/MyDrive/greek-news-nlp/benchmark_summary.csv", index=False)
summary[["model", "method", "accuracy", "macro_f1"]]

,model,method,accuracy,macro_f1
0,Qwen2.5-1.5B-Instruct,"zero-shot, free generation",0.4500,0.3531
1,Qwen2.5-1.5B-Instruct,"zero-shot, letter choice",0.0559,0.0237
2,Qwen2.5-1.5B-Instruct,"zero-shot, first-token",0.5095,0.4276
3,Qwen2.5-1.5B-Instruct,"zero-shot, full-label scoring (sum)",0.4650,0.3549
4,Qwen2.5-1.5B-Instruct,"few-shot 1/class, first-token",0.4854,0.3853
5,GreekBERT (fine-tuned),fine-tuned classifier,0.8200,0.7840
6,Qwen2.5-3B-Instruct,"zero-shot, first-token",0.6264,0.5293


In [21]:
!pip -q install peft bitsandbytes

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 20.7 MB/s eta 0:00:00


In [22]:
import gc, torch
del model
gc.collect(); torch.cuda.empty_cache()

from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

MODEL_ID = "Qwen/Qwen2.5-1.5B-Instruct"
tok = AutoTokenizer.from_pretrained(MODEL_ID)
tok.padding_side = "left"

bnb = BitsAndBytesConfig(
    load_in_4bit=True, bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16, bnb_4bit_use_double_quant=True,
)
base = AutoModelForCausalLM.from_pretrained(MODEL_ID, quantization_config=bnb, device_map="auto")
base = prepare_model_for_kbit_training(base)

lora = LoraConfig(
    r=16, lora_alpha=32, lora_dropout=0.05, task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
)
model = get_peft_model(base, lora)
model.print_trainable_parameters()
print("memory GB:", round(torch.cuda.memory_allocated() / 1e9, 2))

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

trainable params: 18,464,768 || all params: 1,562,179,072 || trainable%: 1.1820
memory GB: 1.71


In [23]:
import numpy as np, random, time
import pandas as pd

TRAIN_PATH = "/content/drive/MyDrive/greek-news-nlp/el_train.jsonl"
train = pd.read_json(TRAIN_PATH, lines=True)

MAX_CHARS_FT = 800
IM_END = tok.convert_tokens_to_ids("<|im_end|>")
PAD = tok.pad_token_id if tok.pad_token_id is not None else tok.eos_token_id

def build_prompt_ft(text):
    messages = [
        {"role": "system", "content": "You are a precise news topic classifier."},
        {"role": "user", "content":
            f"Classify the Greek news article into one IPTC topic.\n\nArticle:\n{text[:MAX_CHARS_FT]}\n\nTopic:"},
    ]
    return tok.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

def encode_example(text, label):
    p = tok(build_prompt_ft(text), add_special_tokens=False).input_ids
    a = tok.encode(label, add_special_tokens=False) + [IM_END]
    return {"input_ids": p + a, "labels": [-100] * len(p) + a}

train_enc = [encode_example(t, l) for t, l in zip(train[TEXT_COL], train[LABEL_COL].astype(str))]
lens = np.array([len(x["input_ids"]) for x in train_enc])
print(len(train_enc), "examples | tokens mean/p95/max:", int(lens.mean()), int(np.percentile(lens, 95)), lens.max())
print("tokens per epoch (M):", round(lens.sum() / 1e6, 2))

ex = train_enc[0]
n_p = sum(1 for t in ex["labels"] if t == -100)
print("TRAINED PART:", repr(tok.decode([t for t in ex["labels"] if t != -100])))
print("PROMPT TAIL :", repr(tok.decode(ex["input_ids"][n_p - 12:n_p])))

3675 examples | tokens mean/p95/max: 685 754 790
tokens per epoch (M): 2.52
TRAINED PART: 'arts, culture, entertainment and media<|im_end|>'
PROMPT TAIL : 'οινό\n\nTopic:<|im_end|>\n<|im_start|>assistant\n'


In [24]:
import torch
from transformers import get_cosine_schedule_with_warmup
from sklearn.metrics import f1_score, accuracy_score

EPOCHS, BATCH_TR, GRAD_ACC, LR = 2, 4, 4, 2e-4
SAVE_DIR = "/content/drive/MyDrive/greek-news-nlp"
random.seed(42); torch.manual_seed(42)

def collate(items):
    L = max(len(x["input_ids"]) for x in items)
    ids = torch.full((len(items), L), PAD)
    lab = torch.full((len(items), L), -100)
    att = torch.zeros((len(items), L), dtype=torch.long)
    for i, x in enumerate(items):
        n = len(x["input_ids"])
        ids[i, :n] = torch.tensor(x["input_ids"])
        lab[i, :n] = torch.tensor(x["labels"])
        att[i, :n] = 1
    return ids.to("cuda"), att.to("cuda"), lab.to("cuda")

first_ids_t = torch.tensor([tok.encode(l, add_special_tokens=False)[0] for l in LABELS], device="cuda")
val_prompts = [build_prompt_ft(t) for t in texts_v]

@torch.no_grad()
def evaluate_val(bs=8):
    model.eval()
    preds, t0 = [], time.time()
    for i in range(0, len(val_prompts), bs):
        enc = tok(val_prompts[i:i+bs], return_tensors="pt", padding=True).to("cuda")
        with torch.autocast("cuda", dtype=torch.float16):
            lg = model(**enc, logits_to_keep=1).logits[:, -1, :]
        preds += [LABELS[b] for b in lg[:, first_ids_t].argmax(-1).tolist()]
    model.train()
    return preds, (time.time() - t0) / len(val_prompts)

# τα LoRA βάρη σε fp32 (απαραίτητο για GradScaler)
for p in model.parameters():
    if p.requires_grad:
        p.data = p.data.float()
params = [p for p in model.parameters() if p.requires_grad]

opt = torch.optim.AdamW(params, lr=LR, weight_decay=0.0)
n_batches = len(train_enc) // BATCH_TR
total_steps = EPOCHS * (n_batches // GRAD_ACC)
sched = get_cosine_schedule_with_warmup(opt, int(0.05 * total_steps), total_steps)
scaler = torch.amp.GradScaler("cuda")

model.config.use_cache = False
model.train()
torch.cuda.reset_peak_memory_stats()
t_start, step = time.time(), 0

for epoch in range(1, EPOCHS + 1):
    order = list(range(len(train_enc)))
    random.Random(42 + epoch).shuffle(order)
    opt.zero_grad(set_to_none=True)
    run_loss, run_n = 0.0, 0
    for b in range(n_batches):
        ids, att, lab = collate([train_enc[j] for j in order[b*BATCH_TR:(b+1)*BATCH_TR]])
        with torch.autocast("cuda", dtype=torch.float16):
            loss = model(input_ids=ids, attention_mask=att, labels=lab).loss
        scaler.scale(loss / GRAD_ACC).backward()
        run_loss += loss.item(); run_n += 1
        if (b + 1) % GRAD_ACC == 0:
            scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(params, 1.0)
            scaler.step(opt); scaler.update()
            opt.zero_grad(set_to_none=True); sched.step(); step += 1
            if step % 25 == 0:
                el = time.time() - t_start
                eta = el / step * (total_steps - step)
                print(f"ep {epoch} step {step}/{total_steps} loss {run_loss/run_n:.3f} "
                      f"elapsed {el/60:.1f} min ETA {eta/60:.1f} min")
                run_loss, run_n = 0.0, 0

    preds_ep, sec = evaluate_val()
    acc_ep = round(accuracy_score(y_val, preds_ep), 4)
    f1_ep = round(f1_score(y_val, preds_ep, average="macro", labels=LABELS, zero_division=0), 4)
    print(f"=== epoch {epoch} | val acc {acc_ep} | macro-F1 {f1_ep} | {sec:.3f} s/article ===")
    model.save_pretrained(f"{SAVE_DIR}/qlora_qwen2.5-1.5b_ep{epoch}")
    out = val.copy(); out["pred"] = preds_ep
    out.to_csv(f"{SAVE_DIR}/val_qlora_qwen2.5-1.5b_ep{epoch}.csv", index=False)

print("total min:", round((time.time() - t_start) / 60, 1),
      "| peak VRAM GB:", round(torch.cuda.max_memory_allocated() / 1e9, 2))

/usr/local/lib/python3.13/dist-packages/torch/_dynamo/eval_frame.py:1263: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


ep 1 step 25/458 loss 1.622 elapsed 6.1 min ETA 105.9 min
ep 1 step 50/458 loss 0.332 elapsed 12.2 min ETA 99.2 min
ep 1 step 75/458 loss 0.279 elapsed 18.2 min ETA 93.0 min
ep 1 step 100/458 loss 0.279 elapsed 24.3 min ETA 87.0 min
ep 1 step 125/458 loss 0.263 elapsed 30.3 min ETA 80.8 min
ep 1 step 150/458 loss 0.223 elapsed 36.4 min ETA 74.7 min
ep 1 step 175/458 loss 0.222 elapsed 42.5 min ETA 68.7 min
ep 1 step 200/458 loss 0.232 elapsed 48.5 min ETA 62.6 min
ep 1 step 225/458 loss 0.192 elapsed 54.6 min ETA 56.5 min
=== epoch 1 | val acc 0.7459 | macro-F1 0.6553 | 0.250 s/article ===


/usr/local/lib/python3.13/dist-packages/torch/_dynamo/eval_frame.py:1263: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


ep 2 step 250/458 loss 0.119 elapsed 64.1 min ETA 53.3 min
ep 2 step 275/458 loss 0.130 elapsed 70.2 min ETA 46.7 min
ep 2 step 300/458 loss 0.090 elapsed 76.2 min ETA 40.2 min
ep 2 step 325/458 loss 0.092 elapsed 82.3 min ETA 33.7 min
ep 2 step 350/458 loss 0.098 elapsed 88.4 min ETA 27.3 min
ep 2 step 375/458 loss 0.087 elapsed 94.5 min ETA 20.9 min
ep 2 step 400/458 loss 0.100 elapsed 100.5 min ETA 14.6 min
ep 2 step 425/458 loss 0.069 elapsed 106.5 min ETA 8.3 min
ep 2 step 450/458 loss 0.088 elapsed 112.6 min ETA 2.0 min
=== epoch 2 | val acc 0.7637 | macro-F1 0.6862 | 0.249 s/article ===
total min: 117.9 | peak VRAM GB: 8.73


In [25]:
from sklearn.metrics import classification_report

new = pd.DataFrame([
    {"model": "Qwen2.5-1.5B-Instruct", "method": "QLoRA r=16, 1 epoch, first-token", "split": "val",
     "accuracy": 0.7459, "macro_f1": 0.6553, "sec_per_article": 0.25, "peak_vram_gb": 8.73,
     "notes": "latency/VRAM measured inside training loop"},
    {"model": "Qwen2.5-1.5B-Instruct", "method": "QLoRA r=16, 2 epochs, first-token", "split": "val",
     "accuracy": 0.7637, "macro_f1": 0.6862, "sec_per_article": 0.249, "peak_vram_gb": 8.73,
     "notes": "118 min training on T4; latency/VRAM measured inside training loop"},
])
summary = pd.concat([summary, new], ignore_index=True)
summary.to_csv("/content/drive/MyDrive/greek-news-nlp/benchmark_summary.csv", index=False)

q = pd.read_csv(f"{SAVE_DIR}/val_qlora_qwen2.5-1.5b_ep2.csv")
print(classification_report(q[LABEL_COL], q["pred"], labels=LABELS, zero_division=0))

                                           precision    recall  f1-score   support

   arts, culture, entertainment and media       0.75      0.77      0.76        79
                  conflict, war and peace       0.47      0.35      0.40        26
                   crime, law and justice       0.81      0.76      0.79        63
disaster, accident and emergency incident       0.69      0.56      0.62        36
            economy, business and finance       0.82      0.80      0.81        99
                                education       0.67      0.90      0.77        31
                              environment       0.77      0.56      0.65        18
                                   health       0.73      0.97      0.84        39
                           human interest       0.61      0.69      0.65        62
                                   labour       0.68      0.62      0.65        21
                    lifestyle and leisure       0.33      0.12      0.18         8
   

In [26]:
import gc, torch, time
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from sklearn.metrics import classification_report, f1_score, accuracy_score

# ελευθέρωση GPU από το Qwen
for name in ["model", "base", "opt", "scaler"]:
    if name in globals():
        del globals()[name]
gc.collect(); torch.cuda.empty_cache()

GB_PATH = "/content/drive/MyDrive/greek-news-nlp/greek-bert-news-classifier"
gb_tok = AutoTokenizer.from_pretrained(GB_PATH)
gb = AutoModelForSequenceClassification.from_pretrained(GB_PATH).to("cuda").eval()
print("id2label:", gb.config.id2label)

MAXLEN_GB = 256
BS = 8
torch.cuda.reset_peak_memory_stats()
preds_gb, t_gb = [], 0.0
for i in range(0, len(texts_v), BS):
    enc = gb_tok(texts_v[i:i+BS], truncation=True, max_length=MAXLEN_GB,
                 padding=True, return_tensors="pt").to("cuda")
    torch.cuda.synchronize(); t0 = time.time()
    with torch.no_grad():
        lg = gb(**enc).logits
    torch.cuda.synchronize(); t_gb += time.time() - t0
    preds_gb += lg.argmax(-1).tolist()

id2label = gb.config.id2label
preds_gb_names = [id2label[p] for p in preds_gb]

print("acc:", round(accuracy_score(y_val, preds_gb_names), 4),
      "| macro-F1:", round(f1_score(y_val, preds_gb_names, average="macro", labels=LABELS, zero_division=0), 4))
print("sec/article:", round(t_gb / len(texts_v), 4), "| peak VRAM GB:", round(torch.cuda.max_memory_allocated() / 1e9, 2))
print(classification_report(y_val, preds_gb_names, labels=LABELS, zero_division=0))

out = val.copy(); out["pred"] = preds_gb_names
out.to_csv("/content/drive/MyDrive/greek-news-nlp/val_greekbert.csv", index=False)

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

id2label: {0: 'arts, culture, entertainment and media', 1: 'conflict, war and peace', 2: 'crime, law and justice', 3: 'disaster, accident and emergency incident', 4: 'economy, business and finance', 5: 'education', 6: 'environment', 7: 'health', 8: 'human interest', 9: 'labour', 10: 'lifestyle and leisure', 11: 'politics', 12: 'religion', 13: 'science and technology', 14: 'society', 15: 'sport', 16: 'weather'}
acc: 0.8158 | macro-F1: 0.7756
sec/article: 0.0162 | peak VRAM GB: 1.57
                                           precision    recall  f1-score   support

   arts, culture, entertainment and media       0.81      0.85      0.83        79
                  conflict, war and peace       0.64      0.69      0.67        26
                   crime, law and justice       0.85      0.84      0.85        63
disaster, accident and emergency incident       0.85      0.78      0.81        36
            economy, business and finance       0.83      0.78      0.80        99
               

In [27]:
summary.loc[summary["model"].str.startswith("GreekBERT"),
            ["accuracy", "macro_f1", "sec_per_article", "peak_vram_gb", "notes"]] = [
    0.8158, 0.7756, 0.0162, 1.57,
    "re-measured in this pipeline (text_clean, max_length=256, batch 8); earlier reference 0.820 / 0.784"]
summary.to_csv("/content/drive/MyDrive/greek-news-nlp/benchmark_summary.csv", index=False)
summary[["model", "method", "accuracy", "macro_f1", "sec_per_article"]]

,model,method,accuracy,macro_f1,sec_per_article
0,Qwen2.5-1.5B-Instruct,"zero-shot, free generation",0.4500,0.3531,0.2910
1,Qwen2.5-1.5B-Instruct,"zero-shot, letter choice",0.0559,0.0237,0.0500
2,Qwen2.5-1.5B-Instruct,"zero-shot, first-token",0.5095,0.4276,0.0500
3,Qwen2.5-1.5B-Instruct,"zero-shot, full-label scoring (sum)",0.4650,0.3549,3.7100
4,Qwen2.5-1.5B-Instruct,"few-shot 1/class, first-token",0.4854,0.3853,0.5250
5,GreekBERT (fine-tuned),fine-tuned classifier,0.8158,0.7756,0.0162
6,Qwen2.5-3B-Instruct,"zero-shot, first-token",0.6264,0.5293,0.1800
7,Qwen2.5-1.5B-Instruct,"QLoRA r=16, 1 epoch, first-token",0.7459,0.6553,0.2500
8,Qwen2.5-1.5B-Instruct,"QLoRA r=16, 2 epochs, first-token",0.7637,0.6862,0.2490


In [28]:
import gc, torch, time
from transformers import AutoModelForCausalLM, BitsAndBytesConfig
from peft import PeftModel
from sklearn.metrics import accuracy_score, f1_score

if "gb" in globals():
    del gb
gc.collect(); torch.cuda.empty_cache()

MODEL_ID = "Qwen/Qwen2.5-1.5B-Instruct"
bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                         bnb_4bit_compute_dtype=torch.float16, bnb_4bit_use_double_quant=True)
base_inf = AutoModelForCausalLM.from_pretrained(MODEL_ID, quantization_config=bnb, device_map="auto")
qmodel = PeftModel.from_pretrained(base_inf, f"{SAVE_DIR}/qlora_qwen2.5-1.5b_ep2").eval()

BS = 8
torch.cuda.reset_peak_memory_stats()
preds_q, t_q = [], 0.0
for i in range(0, len(val_prompts), BS):
    enc = tok(val_prompts[i:i+BS], return_tensors="pt", padding=True).to("cuda")
    torch.cuda.synchronize(); t0 = time.time()
    with torch.no_grad(), torch.autocast("cuda", dtype=torch.float16):
        lg = qmodel(**enc, logits_to_keep=1).logits[:, -1, :]
    torch.cuda.synchronize(); t_q += time.time() - t0
    preds_q += [LABELS[b] for b in lg[:, first_ids_t].argmax(-1).tolist()]

print("acc:", round(accuracy_score(y_val, preds_q), 4),
      "| macro-F1:", round(f1_score(y_val, preds_q, average="macro", labels=LABELS, zero_division=0), 4))
print("sec/article:", round(t_q / len(val_prompts), 4),
      "| peak VRAM GB:", round(torch.cuda.max_memory_allocated() / 1e9, 2))

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

acc: 0.7649 | macro-F1: 0.6882
sec/article: 0.252 | peak VRAM GB: 3.62


In [29]:
m = summary["method"].str.contains("QLoRA r=16, 2 epochs")
summary.loc[m, ["accuracy", "macro_f1", "sec_per_article", "peak_vram_gb", "notes"]] = [
    0.7649, 0.6882, 0.252, 3.62,
    "adapter reloaded for inference (4-bit base + LoRA, batch 8); 118 min training on T4"]
m1 = summary["method"].str.contains("QLoRA r=16, 1 epoch")
summary.loc[m1, ["peak_vram_gb", "notes"]] = [None, "latency measured inside training loop; training VRAM not comparable"]
summary.to_csv("/content/drive/MyDrive/greek-news-nlp/benchmark_summary.csv", index=False)
summary[["model", "method", "accuracy", "macro_f1", "sec_per_article", "peak_vram_gb"]]

,model,method,accuracy,macro_f1,sec_per_article,peak_vram_gb
0,Qwen2.5-1.5B-Instruct,"zero-shot, free generation",0.4500,0.3531,0.2910,4.02
1,Qwen2.5-1.5B-Instruct,"zero-shot, letter choice",0.0559,0.0237,0.0500,4.12
2,Qwen2.5-1.5B-Instruct,"zero-shot, first-token",0.5095,0.4276,0.0500,4.11
3,Qwen2.5-1.5B-Instruct,"zero-shot, full-label scoring (sum)",0.4650,0.3549,3.7100,4.50
4,Qwen2.5-1.5B-Instruct,"few-shot 1/class, first-token",0.4854,0.3853,0.5250,6.05
5,GreekBERT (fine-tuned),fine-tuned classifier,0.8158,0.7756,0.0162,1.57
6,Qwen2.5-3B-Instruct,"zero-shot, first-token",0.6264,0.5293,0.1800,7.45
7,Qwen2.5-1.5B-Instruct,"QLoRA r=16, 1 epoch, first-token",0.7459,0.6553,0.2500,NaN
8,Qwen2.5-1.5B-Instruct,"QLoRA r=16, 2 epochs, first-token",0.7649,0.6882,0.2520,3.62


In [30]:
import gc, time, torch
from sklearn.metrics import accuracy_score, f1_score
from transformers import AutoModelForSequenceClassification

test = pd.read_json("/content/drive/MyDrive/greek-news-nlp/el_test.jsonl", lines=True)
texts_t = test[TEXT_COL].tolist()
y_test = test[LABEL_COL].astype(str).tolist()
assert sorted(set(y_test)) == LABELS, "τα labels του test διαφέρουν από του val"
print(len(test), "test articles")

test_results = {}
def report(name, preds):
    acc = round(accuracy_score(y_test, preds), 4)
    f1 = round(f1_score(y_test, preds, average="macro", labels=LABELS, zero_division=0), 4)
    print(f"{name}: acc {acc} | macro-F1 {f1}")
    out = test.copy(); out["pred"] = preds
    out.to_csv(f"/content/drive/MyDrive/greek-news-nlp/test_{name}.csv", index=False)
    return acc, f1

# --- QLoRA (epoch 2) ---
prompts_t = [build_prompt_ft(t) for t in texts_t]
preds = []
for i in range(0, len(prompts_t), 8):
    enc = tok(prompts_t[i:i+8], return_tensors="pt", padding=True).to("cuda")
    with torch.no_grad(), torch.autocast("cuda", dtype=torch.float16):
        lg = qmodel(**enc, logits_to_keep=1).logits[:, -1, :]
    preds += [LABELS[b] for b in lg[:, first_ids_t].argmax(-1).tolist()]
a, f = report("qlora_ep2", preds)
test_results["qlora_ep2"] = {"accuracy": a, "macro_f1": f}

# --- GreekBERT ---
gb_tok = AutoTokenizer.from_pretrained(GB_PATH)
gb = AutoModelForSequenceClassification.from_pretrained(GB_PATH).to("cuda").eval()
preds = []
for i in range(0, len(texts_t), 8):
    enc = gb_tok(texts_t[i:i+8], truncation=True, max_length=256, padding=True, return_tensors="pt").to("cuda")
    with torch.no_grad():
        preds += [gb.config.id2label[p] for p in gb(**enc).logits.argmax(-1).tolist()]
a, f = report("greekbert", preds)
test_results["greekbert"] = {"accuracy": a, "macro_f1": f}

788 test articles
qlora_ep2: acc 0.7766 | macro-F1 0.6957


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

greekbert: acc 0.8338 | macro-F1 0.775


In [31]:
for n in ["qmodel", "base_inf", "gb"]:
    if n in globals():
        del globals()[n]
gc.collect(); torch.cuda.empty_cache()

def zs_prompt(t, text):
    messages = [
        {"role": "system", "content": "You are a precise news topic classifier."},
        {"role": "user", "content":
            f"Classify the following Greek news article into exactly one of these topics:\n"
            f"{label_list}\n\nArticle:\n{text[:MAX_CHARS]}\n\n"
            f"Answer with the topic name only, exactly as written in the list."},
    ]
    return t.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

def zero_shot_first_token(model_id, texts, bs=8):
    t = AutoTokenizer.from_pretrained(model_id); t.padding_side = "left"
    m = AutoModelForCausalLM.from_pretrained(model_id, torch_dtype=torch.float16, device_map="auto").eval()
    ids = torch.tensor([t.encode(l, add_special_tokens=False)[0] for l in LABELS], device=m.device)
    prompts = [zs_prompt(t, x) for x in texts]
    torch.cuda.reset_peak_memory_stats(); torch.cuda.synchronize()
    t0, preds = time.time(), []
    for i in range(0, len(prompts), bs):
        enc = t(prompts[i:i+bs], return_tensors="pt", padding=True).to(m.device)
        with torch.no_grad():
            lg = m(**enc, logits_to_keep=1).logits[:, -1, :]
        preds += [LABELS[b] for b in lg[:, ids].argmax(-1).tolist()]
    torch.cuda.synchronize()
    sec = (time.time() - t0) / len(prompts)
    vram = torch.cuda.max_memory_allocated() / 1e9
    del m; gc.collect(); torch.cuda.empty_cache()
    return preds, round(sec, 4), round(vram, 2)

for mid, key in [("Qwen/Qwen2.5-1.5B-Instruct", "qwen1.5b_zero_shot"),
                 ("Qwen/Qwen2.5-3B-Instruct", "qwen3b_zero_shot")]:
    p, sec, vram = zero_shot_first_token(mid, texts_t)
    a, f = report(key, p)
    test_results[key] = {"accuracy": a, "macro_f1": f, "sec_per_article": sec, "peak_vram_gb": vram}

res_df = pd.DataFrame(test_results).T
res_df.to_csv("/content/drive/MyDrive/greek-news-nlp/test_results.csv")
print(res_df)

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

qwen1.5b_zero_shot: acc 0.486 | macro-F1 0.4023


Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

qwen3b_zero_shot: acc 0.6294 | macro-F1 0.5183
                    accuracy  macro_f1  sec_per_article  peak_vram_gb
qlora_ep2             0.7766    0.6957              NaN           NaN
greekbert             0.8338    0.7750              NaN           NaN
qwen1.5b_zero_shot    0.4860    0.4023           0.2450          5.09
qwen3b_zero_shot      0.6294    0.5183           0.4721          8.42


In [32]:
import os, numpy as np, pandas as pd
from sklearn.metrics import accuracy_score, f1_score

D = "/content/drive/MyDrive/greek-news-nlp"
names = {
    "GreekBERT (fine-tuned)": "greekbert",
    "Qwen2.5-1.5B + QLoRA": "qlora_ep2",
    "Qwen2.5-3B zero-shot": "qwen3b_zero_shot",
    "Qwen2.5-1.5B zero-shot": "qwen1.5b_zero_shot",
}
dfs = {m: pd.read_csv(f"{D}/test_{k}.csv") for m, k in names.items()}
base = dfs["GreekBERT (fine-tuned)"]
ids = base["document_id"].values
y = base[LABEL_COL].astype(str).values
for m, d in dfs.items():
    assert (d["document_id"].values == ids).all(), m
pred = {m: d["pred"].astype(str).values for m, d in dfs.items()}

def metrics(yt, yp):
    return (accuracy_score(yt, yp),
            f1_score(yt, yp, average="macro", labels=LABELS, zero_division=0))

rng = np.random.default_rng(42)
B = 1000
samples = [rng.integers(0, len(y), len(y)) for _ in range(B)]
boot = {m: np.array([metrics(y[s], p[s]) for s in samples]) for m, p in pred.items()}

lat = {  # sec/article, peak VRAM GB, πού μετρήθηκε
    "GreekBERT (fine-tuned)": (0.0162, 1.57, "val"),
    "Qwen2.5-1.5B + QLoRA": (0.252, 3.62, "val"),
    "Qwen2.5-3B zero-shot": (0.4721, 8.42, "test"),
    "Qwen2.5-1.5B zero-shot": (0.2450, 5.09, "test"),
}
rows = []
for m, p in pred.items():
    a, f = metrics(y, p)
    lo, hi = np.percentile(boot[m], [2.5, 97.5], axis=0)
    rows.append({"model": m, "accuracy": round(a, 4), "acc_95ci": f"{lo[0]:.3f}-{hi[0]:.3f}",
                 "macro_f1": round(f, 4), "f1_95ci": f"{lo[1]:.3f}-{hi[1]:.3f}",
                 "sec_per_article": lat[m][0], "peak_vram_gb": lat[m][1], "latency_measured_on": lat[m][2]})
final = pd.DataFrame(rows)
print(final.to_string(index=False))

d = boot["GreekBERT (fine-tuned)"] - boot["Qwen2.5-1.5B + QLoRA"]
for i, nm in enumerate(["accuracy", "macro-F1"]):
    lo, hi = np.percentile(d[:, i], [2.5, 97.5])
    print(f"GreekBERT - QLoRA, {nm}: {d[:, i].mean():+.3f} (95% CI {lo:+.3f} .. {hi:+.3f})")

# εξαγωγή για το repo
out_dir = f"{D}/results_export"
os.makedirs(f"{out_dir}/predictions", exist_ok=True)
final.to_csv(f"{out_dir}/test_results.csv", index=False)
for m, k in names.items():
    pd.DataFrame({"document_id": ids, "true": y, "pred": pred[m]}).to_csv(
        f"{out_dir}/predictions/test_{k}.csv", index=False)

mask = summary.index.isin([1, 2, 4, 6])
summary.loc[mask, "notes"] = (summary.loc[mask, "notes"].fillna("").astype(str)
    + " | latency measured without cuda.synchronize (underestimated)").str.strip(" |")
summary.to_csv(f"{out_dir}/benchmark_summary_val.csv", index=False)
print(os.listdir(out_dir))

                 model  accuracy    acc_95ci  macro_f1     f1_95ci  sec_per_article  peak_vram_gb latency_measured_on
GreekBERT (fine-tuned)    0.8338 0.810-0.860    0.7750 0.731-0.811           0.0162          1.57                 val
  Qwen2.5-1.5B + QLoRA    0.7766 0.746-0.806    0.6957 0.639-0.732           0.2520          3.62                 val
  Qwen2.5-3B zero-shot    0.6294 0.596-0.665    0.5183 0.473-0.554           0.4721          8.42                test
Qwen2.5-1.5B zero-shot    0.4860 0.452-0.520    0.4023 0.351-0.438           0.2450          5.09                test
GreekBERT - QLoRA, accuracy: +0.058 (95% CI +0.029 .. +0.088)
GreekBERT - QLoRA, macro-F1: +0.083 (95% CI +0.032 .. +0.141)
['predictions', 'test_results.csv', 'benchmark_summary_val.csv']
